In [1]:
DATA = "../data/auto_mpg.csv"
OUTPUT = "../submission"
ESTIMATOR = f"{OUTPUT}/estimator.pkl"

In [2]:
def load_data():

    import pandas as pd

    dataset = pd.read_csv(DATA)
    dataset = dataset.dropna()
    dataset["Origin"] = dataset["Origin"].map(
        {1: "USA", 2: "Europe", 3: "Japan"},
    )
    y = dataset.pop("MPG")
    x = dataset.copy()

    return x, y
  

In [3]:
def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=123456,
    )
    return x_train, x_test, y_train, y_test

In [5]:
def make_pipeline(estimator):

    from sklearn.compose import ColumnTransformer
    from sklearn.feature_selection import SelectKBest, f_regression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    transformer = ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(dtype="int"), ["Origin"]),
        ],
        remainder=StandardScaler(),
    )

    selectkbest = SelectKBest(score_func=f_regression)

    pipeline = Pipeline(
        steps=[
            ("tranformer", transformer),
            ("selectkbest", selectkbest),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline  

In [6]:
def make_grid_search(estimator, param_grid, cv=5):

    from sklearn.model_selection import GridSearchCV

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="neg_mean_absolute_error",
    )

    return grid_search

In [7]:
def load_estimator():

    import os
    import pickle

    if not os.path.exists(ESTIMATOR):
        return None
    with open(ESTIMATOR, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [8]:
def save_estimator(estimator, x, y):

    import os
    import pickle

    from sklearn.metrics import mean_absolute_error

    best_estimator = load_estimator()

    if best_estimator is not None:

        saved_mae = mean_absolute_error(y_true=y, y_pred=best_estimator.predict(x))

        current_mae = mean_absolute_error(y_true=y, y_pred=estimator.predict(x))

        if saved_mae < current_mae:
            estimator = best_estimator

    with open(ESTIMATOR, "wb") as file:
        pickle.dump(estimator, file)

In [9]:
def train_linear_regression():

    import os

    from sklearn.linear_model import LinearRegression

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=LinearRegression(),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "selectkbest__k": range(1, len(x_train.columns) + 1),
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


train_linear_regression()